In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# external imports
from pathlib import Path
import os
from dotenv import load_dotenv
from torch_geometric.data import Batch
import torch
import random

In [3]:
# internal imports
from primaite.network.generator import NetworkGenerator
from primaite.agents.aegis.modules.openai import OpenAIClient

2024-09-10 14:52:43.470878: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-10 14:52:43.502857: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-10 14:52:44.071363: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/sam/projects/PrimAITE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sam/projects/PrimAITE/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected name

In [4]:
# constants
DATASET_SIZE = 20
CORPUS_GRAPH_SIZES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
TRAINING_CONFIG_PATH = Path("../agents") / "training_configs" / "do_nothing.yaml"
SERVICE_NAMES = ["HTTP", "SSH", "FTP"]
PORTS_LIST = ["80", "22", "21"]

In [ ]:
load_dotenv() 
open_ai_key = os.getenv('OPEN_AI_KEY')

openai = OpenAIClient(api_key=open_ai_key)

In [ ]:
for corpus_graph_size in CORPUS_GRAPH_SIZES:
    corpus = []

    laydown_root_dir = Path("../data") / "notebook_generated" / "diverse_datasets" / f"corpus_size_{corpus_graph_size}.yaml"
    laydown_root_dir.mkdir(parents=True, exist_ok=True)

    for i in range(DATASET_SIZE // len(CORPUS_GRAPH_SIZES)):
        laydown_save_path = laydown_root_dir / f"{i}.yaml"
        generator = NetworkGenerator(graph_size=corpus_graph_size, training_config_path=TRAINING_CONFIG_PATH, random_seed=None, services=SERVICE_NAMES, ports=PORTS_LIST, pretrained_llm=openai)
        output = generator.run_end_to_end(laydown_save_path=laydown_save_path)
        corpus.append(output)

    batch = Batch.from_data_list(corpus)
    torch.save(batch, Path("../data") / "notebook_generated" / "diverse_datasets" / f"corpus_size_{corpus_graph_size}.pt")